In [4]:
import pandas as pd
import numpy as np
import seaborn as sns


In [5]:
ercot = pd.read_csv("ercot_artifex.csv")
ercot["time_utc"] = pd.to_datetime(ercot["datetime_utc"], format="%m/%d/%y %H:%M")

zones = ["LZ_HOUSTON", "LZ_NORTH", "LZ_SOUTH", "LZ_WEST"]

LZ = ercot.query("type == 'LZ' & spn in @zones")
wide = LZ.pivot(index="time_utc", columns="spn", values="price").reset_index()

wide = (
    wide
    .assign(
        spread=lambda x: x[zones].max(axis=1) - x[zones].min(axis=1),
        month=lambda x: x["time_utc"].dt.month,
        prepost=lambda x: np.where(x["time_utc"] < "2025-12-05", "Pre-RTCB", "Post-RTCB")
    )
    .query("month <= 7")
)
wide.head()

ModuleNotFoundError: No module named 'pandas.core.reshape.reshape'

In [ ]:
cutoff = wide.query("prepost == 'Pre-RTCB'")["spread"].quantile(0.95)
print("Cutoff ($/MWh):", cutoff)

unusual = (
    wide
    .assign(big=lambda x: x["spread"] > cutoff)
    .groupby(["month", "prepost"], as_index=False)
    .agg(pct_big=("big", "mean"))
    .assign(pct_big=lambda x: x["pct_big"] * 100)
)

unusual.head()

Cutoff ($/MWh): 62.94


,month,prepost,pct_big
0,1,Post-RTCB,5.477151
1,1,Pre-RTCB,7.892954
2,2,Post-RTCB,1.822917
3,2,Pre-RTCB,8.928571
4,3,Post-RTCB,2.688172


In [1]:

ax = sns.barplot(
    data=unusual,
    x="month",
    y="pct_big",
    hue="prepost",
    hue_order=["Pre-RTCB", "Post-RTCB"]
)
ax.set(
    title="Monthly Amounts of \"Unusually\" Large Spatial Price Spreads Before and After RTC+B",
    xlabel="Month",
    ylabel="% of 15-minute intervals with unusually large spread",
    xticklabels=["Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul"]
)
sns.move_legend(ax, "upper right", title="Period")

NameError: name 'sns' is not defined